# T00 · Extending MACE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neutrino155/MACE_tutorials/blob/main/MACE_extensions/T00-Extending-MACE.ipynb)

**Lennard-Jones Centre · University of Cambridge · Atomic-Scale Modelling**

A theory-first map of four ways to extend MACE. The main hands-on workshop is in Practice I and II; use this overview before the closing extension demonstrations.

## Learning objectives

By the end of this notebook, you should be able to:

- identify what information a baseline MACE model is missing;
- place a new input, readout or physical term in the architecture;
- name the data, software and validation checks needed for an extension.

## 1. Start with the baseline architecture

MACE maps species, positions and cell information to a scalar energy. Radial and angular features describe neighbour geometry; equivariant interaction and product blocks build local many-body features; an invariant readout produces atomic energy contributions. Forces are energy gradients with respect to positions.

The sketch below is the starting point. An extension keeps this path where possible and adds the smallest mechanism needed to answer the scientific question.

![MACE architecture with embedding, interaction and product stages and invariant readout](./figures/mace_theory_architecture.png)

## 2. Four extension patterns

Ask: **what extra information makes the target well-defined?** Each example adds that information at a different point.

| Extension | New information | Where it enters | Main contract |
|---|---|---|---|
| **MACEField** | Uniform electric field | Field-conditioned equivariant features and scalar energy | Rotate field and structure together; derive polarization from energy |
| **AtomicDipolesMACE** | Molecular dipole | Equivariant vector readout | Polar-vector covariance; state charge and origin |
| **Magnetic MACE** | Site magnetic moments | Moment-dependent features before energy readout | State rotation, parity, time reversal and units |
| **MACELES** | Long-range electrostatics | Latent readouts plus explicit energy term | Specify charge and boundary conditions |

Feed a variable into the energy model when it changes the energy surface. Use a symmetry-matched readout for a directly predicted property, and an explicit energy branch for a known nonlocal interaction.

## 3. Energy or direct property?

If a property is a derivative of energy, predicting one shared energy gives a consistency relation:

$$\mathbf F_i=-\frac{\partial E}{\partial \mathbf R_i},\qquad
\mathbf P=-\frac{1}{\Omega}\frac{\partial E}{\partial\mathbf E}.$$

A direct-property head can be simpler when that property is the only target, but it does not enforce the corresponding derivative identity. Choose the route from the scientific question and labels, and identify which outputs are direct predictions and which come from energy derivatives.

## 4. MACEField: the field changes the energy

MACEField injects the external polar vector into equivariant hidden features and predicts a field-dependent scalar energy, $E(\mathbf R,\mathbf E)$. Polarization, forces and mixed responses can then be derived from that energy. In a periodic cell, this implementation reports polarization density; its Born charges and susceptibility use the same coordinate and field derivatives.

Hysteresis requires following a metastable minimum through a field sweep. Independent minimization at each field finds equilibrium and loses the switching history. T01 compares the analytic reference loop with the trained model's relaxed path.

The four examples keep the species-and-geometry trunk as a starting point. Each diagram marks where the new input, feature or physical term enters.

### MACEField

![MACEField architecture: electric field enters the equivariant interaction stack](./figures/architecture_macefield.png)

## 5. AtomicDipolesMACE: read out an equivariant vector

A dipole is a rank-one polar tensor, so the model retains vector irreducible representations and attaches an equivariant readout to node features. Under a rotation $Q$, it should satisfy $\boldsymbol\mu(Q\mathbf R)=Q\boldsymbol\mu(\mathbf R)$.

For a neutral molecule, translation leaves the total dipole unchanged; a charged system requires a stated origin. Atomic dipole contributions are internal model quantities, not unique partial charges.

### AtomicDipolesMACE

![AtomicDipolesMACE architecture: equivariant vector readout from node features](./figures/architecture_atomicdipoles.png)

## 6. Magnetic MACE: moments are state variables

Magnetic MACE learns a scalar energy $E(\mathbf R,\mathbf m)$ conditioned on site moments. Moment-dependent features enter the interaction and product blocks before the energy readout; forces and magnetic forces are derivatives at fixed inputs:

$$\mathbf F_i=-\left.\frac{\partial E}{\partial\mathbf R_i}\right|_{\mathbf m},\qquad
\mathbf f_i^m=-\left.\frac{\partial E}{\partial\mathbf m_i}\right|_{\mathbf R}.$$

State moment units and transformation conventions. A self-consistent wrapper that relaxes moments changes the derivative being evaluated.

### Magnetic MACE

![Magnetic MACE architecture: ](./figures/architecture_magnetic.png)

## 7. MACELES: add a nonlocal energy term

MACELES keeps a local MACE energy and adds an electrostatic energy evaluated from learned latent quantities:

$$E_{\mathrm{total}}=E_{\mathrm{local}}+E_{\mathrm{LES}}.$$

Readouts predict charges and optional multipoles or response parameters; an LES solver combines them with positions and the cell. Forces come from the total energy. Latent charges are not automatically a unique chemical charge analysis.

### MACELES

![MACELES architecture: ](./figures/architecture_maceles.png)

## 8. MACEField: trace one extension through the codebase

Start with a physical change: the energy now depends on an applied electric field, so the model must accept field labels and return responses consistent with that energy. Follow the change through four code areas. The snippets are abbreviated excerpts from MACE-Field itself; each source link opens the corresponding code at the revision used for these examples.

### 8.1 Data contract: how MACEField reads the labels

<div style="background:#eef4f7;border-left:5px solid #1f6f8b;padding:12px 16px;border-radius:6px;margin:12px 0">
<b>Question:</b> Which labels are graph-level properties and which are per-atom arrays? MACEField defines default label names, routes them from ExtXYZ, then checks their tensor shapes as it builds a batch.
</div>

**MACEField source:** [default label names](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/tools/default_keys.py#L6-L16) · [ExtXYZ key routing](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/data/utils.py#L49-L80)

```python
# DefaultKeys in mace/tools/default_keys.py
ELECTRIC_FIELD = "REF_electric_field"
POLARIZATION = "REF_polarization"
BECS = "REF_becs"
POLARIZABILITY = "REF_polarizability"

# update_keyspec_from_kwargs in mace/data/utils.py
infos = [..., "electric_field_key", "polarization_key", "polarizability_key", ...]
arrays = [..., "becs_key", ...]
```

The loader treats the field, polarization and polarizability as graph properties; BECs are per-atom arrays. The code checks shapes but does not store units in the tensors, so the data pipeline and ExtXYZ metadata must state the unit convention used for the labels.

**Shape checks in MACEField:** [AtomicData validation](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/data/atomic_data.py#L130-L145)

```python
assert electric_field is None or electric_field.shape[-1] == 3
assert polarization is None or polarization.shape[-1] == 3
assert becs is None or becs.shape == (num_nodes, 3, 3)
assert polarizability is None or polarizability.shape == (1, 3, 3)
```

### 8.2 Model: inject the field into equivariant features

<div style="background:#f0f6f1;border-left:5px solid #4b8b63;padding:12px 16px;border-radius:6px;margin:12px 0">
<b>Question:</b> Where can the new input influence the energy? MACEField inherits the MACE energy model and couples the polar-vector field to hidden features between interaction layers.
</div>

**MACEField source:** [field adapter construction](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/extensions.py#L157-L182) · [per-atom field and forward-pass injection](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/extensions.py#L311-L378)

```python
# MACEField.__init__: the field has polar-vector irreps, 1o
self.field_feats = torch.nn.ModuleList(
    FullyConnectedTensorProduct(
        irreps_in1=hidden_irreps,
        irreps_in2=o3.Irreps("1o"),
        irreps_out=hidden_irreps,
        cueq_config=cueq_config,
    )
    for _ in range(self.num_interactions - 1)
)

# MACEField.forward: couple the graph field to each atom's features
per_atom_electric_field = electric_field[data["batch"].to(torch.long)]
delta_node_feats = field_feat(node_feats, per_atom_electric_field)
node_feats = node_feats - field_linear(delta_node_feats)
```

The base energy model is in `mace/modules/models.py`; MACEField's field coupling is in `mace/modules/extensions.py`. Shared geometric interaction and product blocks are in `mace/modules/blocks.py`. Change shared blocks only when the common feature construction must change; an additional input can often enter through a model-specific adapter.

### 8.3 Train the physics: differentiate the shared energy

<div style="background:#fbf4e9;border-left:5px solid #c78a32;padding:12px 16px;border-radius:6px;margin:12px 0">
<b>Question:</b> Which outputs should remain tied to one energy? MACEField differentiates its field-dependent energy for polarization, then differentiates polarization with respect to positions and field to obtain BECs and susceptibility.
</div>

**MACEField source:** [response derivatives requested in the model](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/extensions.py#L444-L481) · [autograd implementation of polarization](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/utils.py#L779-L808)

```python
# get_polarization in mace/modules/utils.py
polarization = torch.autograd.grad(
    outputs=[energy],
    inputs=[electric_field],
    grad_outputs=[torch.ones_like(energy)],
    retain_graph=True,
    create_graph=create_graph,
    allow_unused=True,
)[0]
if polarization is None:
    polarization = torch.zeros_like(electric_field)
return -polarization  # P is -dE/dE_field, before division by volume
```

`create_graph=True` retains the derivative graph so training can differentiate response errors again. The next autograd call takes derivatives of each polarization component with respect to both positions and field.

**MACEField source:** [joint BEC and polarizability derivatives](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/utils.py#L1006-L1084) · [response normalization in MACEField](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/extensions.py#L472-L481) · [polarization volume scaling](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/extensions.py#L531-L541)

```python
# Core of _get_becs_and_polarizability_loop in mace/modules/utils.py
for d in range(3):
    gradients = torch.autograd.grad(
        outputs=[polarization[:, d]],
        inputs=[positions, electric_field],
        grad_outputs=[torch.ones_like(polarization[:, d])],
        retain_graph=True,
        create_graph=create_graph,
        allow_unused=True,
    )
    becs_list.append(gradients[0])
    polarizability_list.append(gradients[1])
```

This differentiates the field-derived dipole-like response with respect to atomic positions for BECs and with respect to field for susceptibility. MACEField divides the reported polarization by volume, and the susceptibility by volume and vacuum permittivity.

`UniversalFieldLoss` combines energy, forces, polarization, BEC and polarizability errors with separate weights. Its polarization term has an extra step because polarization in a periodic solid is **multi-valued**: values separated by an integer polarization quantum describe the same state. The loss folds the prediction onto the nearest equivalent branch using the cell, then compares the folded residual. Its default `normalized_metric` mode measures that residual with a cell-aware metric.

**MACEField source:** [polarization loss mode and weights](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/loss.py#L743-L758) · [branch folding and loss use](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/loss.py#L940-L1003) · [nearest polarization-quantum image](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/tools/utils.py#L310-L335) · [cell-aware distance](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/modules/loss.py#L99-L124)

```python
# UniversalFieldLoss.forward: fold periodic polarization before scoring it
dP_folded, c_folded = fold_polarization(
    pred["polarization"], ref["polarization"], ref["cell"]
)
polarization_residual = config_polarization_weight * (
    normalized_metric_polarization_distance(c_folded, ref["cell"])
)

# Configure the coupled property losses
loss_fn = UniversalFieldLoss(
    polarization_weight=1.0,
    becs_weight=0.1,
    polarizability_weight=0.01,
    polarization_loss_mode="normalized_metric",
    polarizability_loss_mode="standardized_symmetric_huber",
)
```

### 8.4 Fine-tune and use it: transfer weights, then request responses

<div style="background:#f4eff8;border-left:5px solid #8264a2;padding:12px 16px;border-radius:6px;margin:12px 0">
<b>Question:</b> How does a pretrained model become usable with the extension? The model factory copies compatible MACE weights into the MACEField model; the calculator passes the applied field and requested response flags into inference.
</div>

**MACEField source:** [foundation-weight transfer call](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/tools/model_script_utils.py#L235-L249) · [weight-transfer helper](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/tools/finetuning_utils.py#L196-L220)

```python
if model_foundation is not None:
    model = load_foundations_elements(
        model,
        model_foundation,
        z_table,
        load_readout=args.foundation_model_readout,
        max_L=args.max_L,
        default_dtype=dtype_dict.get(args.default_dtype, torch.float64),
    )
```

The MACEField-specific adapters are part of the target model; the transfer helper initializes compatible backbone parameters, while the training configuration controls which parameters are updated.

**MACEField source:** [calculator inputs and response forwarding](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/calculators/mace.py#L125-L150) · [field and response arguments passed to the model](https://github.com/mdi-group/mace-field/blob/1bd205048383a0cae6982cccd687e1837aea717a/mace/calculators/mace.py#L755-L796)

```python
if self.is_macefield:
    batch_dict["electric_field"] = self._resolve_electric_field(atoms).to(
        device=batch_dict["positions"].device,
        dtype=batch_dict["positions"].dtype,
    )
    model_kwargs.update(
        compute_polarization=field_compute_polarization,
        compute_becs=bool(field_compute_becs),
        compute_polarizability=bool(field_compute_polarizability),
        electric_field=batch_dict["electric_field"],
    )
out = model(batch_dict, **model_kwargs)
```

Validate the outputs with finite differences, symmetry checks, held-out response errors and a field scan; do not infer a correct switching loop from a low aggregate loss alone.

<details>
<summary><b>Code-by-code checklist</b></summary>

| Step | Change to make | MACEField path |
|---|---|---|
| **1 · Define labels** | Generate or convert targets; document keys, units, shapes and weights. | `MACE_extensions/scripts/prepare_teaching_data.py`; `mace/data/utils.py`; `mace/data/atomic_data.py` |
| **2 · Expose the model** | Add arguments and route the model name, labels and derivative requests into the constructor and training loop. | `mace/tools/arg_parser.py`; `mace/tools/model_script_utils.py`; `mace/cli/run_train.py` |
| **3 · Extend the forward pass** | Keep the scalar energy; inject the field through equivariant feature adapters. Add shared blocks only if common feature construction changes. | `mace/modules/models.py`; `mace/modules/extensions.py`; shared `mace/modules/blocks.py` |
| **4 · Derive and fit responses** | Compute forces, stress, polarization, BECs and polarizability from the energy; add or wire weighted losses for the new labels. | `mace/modules/utils.py`; `mace/modules/loss.py`; `mace/cli/run_train.py` |
| **5 · Transfer and deploy** | Initialize/freeze the intended parameters; request field-dependent outputs from the calculator and evaluation CLI. | `mace/tools/finetuning_utils.py`; `mace/calculators/mace.py`; `mace/cli/eval_configs.py` |
| **6 · Validate the claim** | Check symmetries, finite-difference derivatives, held-out response errors and behavior over the field range. | `tests/extensions/macefield/test_macefield.py`; `MACE_extensions/scripts/` |

</details>

The detailed [MACEField notebook](T01-MACEField.ipynb) follows this path with runnable data, training and response plots.

## 9. A reusable extension workflow

<details>
<summary><b>Implementation checklist</b></summary>

1. Define the physical target, inputs and range of validity.
2. Choose an energy condition, derivative, direct readout or explicit energy term.
3. Specify symmetry, units, periodicity and data keys.
4. Make the smallest architecture change and connect it to the model factory, parser, loss and calculator.
5. Check shapes and labels on a few frames before training.
6. Validate with held-out states, symmetry transforms, finite differences and a matched baseline.
7. Scan the trained model over the new input; separate model predictions from reference labels.

</details>

## 10. Validate the claimed behaviour

- **Transformations:** rotate the structure and all vector/tensor inputs together; check the output representation.
- **Derivatives:** compare energy gradients with finite differences over a stable range of step sizes.
- **Limits:** remove the field, zero the moments or separate fragments as appropriate.
- **Conservation:** check charge, acoustic sum rules, energy consistency and cell conventions.
- **Matched control:** use the same data, split and fitting protocol, changing only the extension.
- **State-space scan:** plot the trained model over the added variable, not only the reference data.

## 11. Common failure patterns

<details>
<summary><b>Symptoms and checks</b></summary>

| Symptom | Check |
|---|---|
| An output rotates incorrectly | Verify parity and run a joint rotation test. |
| The new input barely changes energy | Confirm it reaches the forward path; scan a fixed geometry. |
| Response values look unreasonable | Check units, volume normalization and finite differences. |
| Training loss is low but the target effect is absent | Inspect held-out state-space scans and per-target errors. |
| Latent variables are treated as observables | Validate energy and forces; state the constraints and ambiguity. |
| A fixed-input derivative is called relaxed | Include the response of the self-consistent optimization. |

A smooth curve alone does not validate a physical claim.

</details>

## 12. Design a fifth extension

Choose a target such as an external magnetic field, local stress, charge-conditioned energy, density or long-range dispersion. Make a short proposal:

1. What input or output is missing, and how does it transform?
2. Where does it enter the architecture, and which data and software paths change?
3. What symmetry test, derivative or limiting-case test, and matched control could falsify the design?

## Continue in this repository

The core route is [MACE Practice I](../MACE_in_practice_I/T01-MACE-Practice-I.ipynb) and [MACE Practice II](../MACE_in_practice_II/T02-MACE-Practice-II.ipynb). Optional advanced material: [MACE Theory](../MACE_advanced/T03-MACE-Theory.ipynb), [MLIP applications](../MACE_advanced/T04-MLIP-Apps.ipynb) and [molecular NEB](../MACE_advanced/T05-MACE-neb.ipynb). Continue with the four extension examples in this folder.

## Notebook acknowledgements

The source notebooks for Practice I, Practice II, the two advanced application examples and MACE Theory were made by **Ioan Magdău, Ilyes Batatia and Will Baldwin**, and further refined by **Ioan Magdău and Alin Elena**. The MACE Theory notebook credits **Will Baldwin** as its developer and identifies **Ilyes Batatia’s** developer tutorial as its basis. The MACE extension notebooks were made by **Bradley Martin**. The workshop team is **Bradley Martin, Joe Hart and Isaac Parker**.